JUEGO DE SERPIENTE
Concepto: que la serpiente sepa en que direcciones tomar para que pueda comer mas manzanas sin morir
la serpiente muere cuando choca con los limites del campo o cuando se choca consigo mismo

In [1]:
import numpy as np
import pickle
import random
import matplotlib.pyplot as plt
from IPython.display import clear_output

Creamos el tablero y las condiciones de como funciona el movimiento de la serpiente "snake"

In [ ]:
FILAS = 10
COLS = 10

##colocamos valores a los campos
ARRIBA = 0
ABAJO = 1
IZQUIERDA = 2
DERECHA = 3

DIRS = {
    ARRIBA:    (-1, 0),
    ABAJO:     ( 1, 0),
    IZQUIERDA: ( 0,-1),
    DERECHA:   ( 0, 1),
}

##la clase snake
class Snake:
    def __init__(self):
        self.reset()

    def reset(self):
        self.cuerpo = [(5, 5), (5, 4), (5, 3)]
        self.direccion = DERECHA
        self.vivo = True
        self.puntos = 0
        self._colocar_manzana()
        return self._get_estado()

    def _colocar_manzana(self):
##colocamos una manzanita a los campos libres del tablero
        libres = [(r, c) for r in range(FILAS) for c in range(COLS)
                  if (r, c) not in self.cuerpo]
        self.manzana = random.choice(libres)

    def _get_estado(self):
        cabeza = self.cuerpo[0]
        ##colocamos una regla fundamental ya que en el juego no se puede mover a su lado opuesto donde se encuentra la serpiente
        opuestos = {ARRIBA: ABAJO, ABAJO: ARRIBA, IZQUIERDA: DERECHA, DERECHA: IZQUIERDA}

        peligros = []
        for d, (dr, dc) in DIRS.items():
            ## nr: new row, nc:new columna nos indica la nueva posicion de la cabeza de la serpiente y con eso verificamos si 
            ## esa posicion se choco o no
            nr, nc = cabeza[0] + dr, cabeza[1] + dc
            choque = (nr < 0 or nr >= FILAS or nc < 0 or nc >= COLS
                      or (nr, nc) in self.cuerpo)
            peligros.append(int(choque))

        man_arriba  = int(self.manzana[0] < cabeza[0])
        man_abajo   = int(self.manzana[0] > cabeza[0])
        man_izq     = int(self.manzana[1] < cabeza[1])
        man_der     = int(self.manzana[1] > cabeza[1])

        return tuple(peligros + [self.direccion, man_arriba, man_abajo, man_izq, man_der])

    def step(self, accion):
        ##colocamos una regla fundamental ya que en el juego no se puede mover a su lado opuesto donde se encuentra la serpiente
        ##podriamos ponerlo como variable global pero para mayor entendimiento
        opuestos = {ARRIBA: ABAJO, ABAJO: ARRIBA, IZQUIERDA: DERECHA, DERECHA: IZQUIERDA}
        if accion != opuestos[self.direccion]:
            self.direccion = accion

        dr, dc = DIRS[self.direccion]
        cabeza = self.cuerpo[0]
        nueva_cabeza = (cabeza[0] + dr, cabeza[1] + dc)
   ##si es que llega a tocar con algun limite se muere la serpiente
        if (nueva_cabeza[0] < 0 or nueva_cabeza[0] >= FILAS or
            nueva_cabeza[1] < 0 or nueva_cabeza[1] >= COLS or
            nueva_cabeza in self.cuerpo):
            self.vivo = False
            return self._get_estado(), -10, True

        self.cuerpo.insert(0, nueva_cabeza)
  ##le damos la recompensa por comer la manzana
        if nueva_cabeza == self.manzana:
            self.puntos += 1
            self._colocar_manzana()
            recompensa = 10
        else:
            self.cuerpo.pop()
            cabeza_ant = self.cuerpo[0]
            dist_ant = abs(cabeza_ant[0] - self.manzana[0]) + abs(cabeza_ant[1] - self.manzana[1])
            dist_nueva = abs(nueva_cabeza[0] - self.manzana[0]) + abs(nueva_cabeza[1] - self.manzana[1])
            recompensa = 0.5 if dist_nueva < dist_ant else -0.5

        return self._get_estado(), recompensa, not self.vivo

In [ ]:
class AgenteQLearning:
    ## alpha= tasa de aprendizaje
    ## gamma= factor de descuento mide cuanto le importa la recompensa
    ## epsilon= tasa de exploracion
    ## epsilon_min y epsilon_decay= son nuestra guias al principio que van a ir disminuyendo para que ya no explore al azar
    def __init__(self, alpha=0.1, gamma=0.9, epsilon=1.0, epsilon_min=0.05, epsilon_decay=0.995):
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.q_table = {}

    def _get_q(self, estado, accion):
        return self.q_table.get((estado, accion), 0.0)

    def elegir_accion(self, estado):
        ##si el numero ramdon es menor a la exploracion se manda un ramdon si no se veridica en los estados
        ## la funcion de valor y toma una ruta
        if random.random() < self.epsilon:
            return random.randint(0, 3)
        valores = [self._get_q(estado, a) for a in range(4)]
        return int(np.argmax(valores))

    def actualizar(self, estado, accion, recompensa, siguiente_estado, terminado):
        ##actualizamos los estados simplemente y su recompensa
        q_actual = self._get_q(estado, accion)
        if terminado:
            objetivo = recompensa
        else:
            max_q_siguiente = max(self._get_q(siguiente_estado, a) for a in range(4))
            objetivo = recompensa + self.gamma * max_q_siguiente
        self.q_table[(estado, accion)] = q_actual + self.alpha * (objetivo - q_actual)

    def decaer_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

Probamos el agente, por el tiempo solo seran 5000 episodios

In [18]:
EPISODIOS = 5000

agente = AgenteQLearning()
env = Snake()

historial_puntos = []
historial_epsilon = []

for ep in range(1, EPISODIOS + 1):
    estado = env.reset()
    pasos = 0

    while env.vivo and pasos < 500:
        accion = agente.elegir_accion(estado)
        siguiente_estado, recompensa, terminado = env.step(accion)
        agente.actualizar(estado, accion, recompensa, siguiente_estado, terminado)
        estado = siguiente_estado
        pasos += 1

    agente.decaer_epsilon()
    historial_puntos.append(env.puntos)
    historial_epsilon.append(agente.epsilon)

    if ep % 500 == 0:
        promedio = np.mean(historial_puntos[-500:])
        print(f"Ep {ep:5d} | Promedio últimos 500: {promedio:.2f} | ε = {agente.epsilon:.3f} | Estados conocidos: {len(agente.q_table)}")

print("\nEntrenamiento terminado.")

Ep   500 | Promedio últimos 500: 3.16 | ε = 0.082 | Estados conocidos: 649
Ep  1000 | Promedio últimos 500: 10.36 | ε = 0.050 | Estados conocidos: 848
Ep  1500 | Promedio últimos 500: 10.77 | ε = 0.050 | Estados conocidos: 909
Ep  2000 | Promedio últimos 500: 11.89 | ε = 0.050 | Estados conocidos: 950
Ep  2500 | Promedio últimos 500: 11.71 | ε = 0.050 | Estados conocidos: 970
Ep  3000 | Promedio últimos 500: 11.81 | ε = 0.050 | Estados conocidos: 986
Ep  3500 | Promedio últimos 500: 11.43 | ε = 0.050 | Estados conocidos: 989
Ep  4000 | Promedio últimos 500: 11.16 | ε = 0.050 | Estados conocidos: 999
Ep  4500 | Promedio últimos 500: 11.20 | ε = 0.050 | Estados conocidos: 1005
Ep  5000 | Promedio últimos 500: 11.03 | ε = 0.050 | Estados conocidos: 1008

Entrenamiento terminado.


podemos ver que conocio 1008 estados al finalizar los episodios

Guardamos los estados

In [38]:
with open('agente_snake.pickle', 'wb') as f:
    pickle.dump(agente.q_table, f)

print(f"Agente guardado. Estados en Q-table: {len(agente.q_table)}")

Agente guardado. Estados en Q-table: 1008


In [40]:
def dibujar_tablero(env):
    grilla = [['.' for _ in range(COLS)] for _ in range(FILAS)]
    for r, c in env.cuerpo[1:]:
        grilla[r][c] = 'o'
    cr, cc = env.cuerpo[0]
    grilla[cr][cc] = 'O'
    mr, mc = env.manzana
    grilla[mr][mc] = 'M'
    print('+' + '-' * COLS + '+')
    for fila in grilla:
        print('|' + ''.join(fila) + '|')
    print('+' + '-' * COLS + '+')
    print(f'Puntos: {env.puntos}')


env_test = Snake()
estado = env_test.reset()
pasos = 0

while env_test.vivo and pasos < 300:
    accion = agente.elegir_accion(estado)
    estado, _, terminado = env_test.step(accion)
    pasos += 1

clear_output(wait=True)
dibujar_tablero(env_test)
print(f'\nPartida terminada en {pasos} pasos.')

+----------+
|..........|
|..........|
|..........|
|..........|
|....oooo..|
|....ooOo..|
|.....ooo.M|
|..........|
|..........|
|..........|
+----------+
Puntos: 8

Partida terminada en 70 pasos.
